# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


The first run downloads three pretrained models from Hugging Face. Internet access to the model hub and its download hosts is required. Run the setup and Parts A–C before Tasks 2 and 3, which reuse the classifier and FLAN-T5 model.

In [ ]:
%pip install -q "transformers>=4.45,<5" sentencepiece torch

In [ ]:
import torch
from transformers import pipeline, set_seed

set_seed(42)
torch.set_num_threads(2)
print("Libraries loaded successfully!")

# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [ ]:
generator = pipeline("text-generation", model="distilgpt2", device=-1)
prompt = "Natural language processing is"
output = generator(
    prompt, max_new_tokens=30, do_sample=False,
    pad_token_id=generator.tokenizer.eos_token_id,
)
print(output[0]["generated_text"])

# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
model.eval()

prompt = "Translate to French: I love natural language processing."
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)
text = "This NLP lab is easy and interesting."
print(classifier(text))

# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

| Task | Most suitable family | Reason |
|---|---|---|
| Sentiment classification | Encoder-only | Builds a contextual representation of the input for classification. |
| Machine translation | Encoder–decoder | Encodes the source text and generates a target-language sequence. |
| Open-ended text generation | Decoder-only | Generates a continuation one token at a time. |
| Named Entity Recognition | Encoder-only | Uses context on both sides of tokens to identify entity labels. |
| Text summarization | Encoder–decoder | Reads a source passage and generates a shorter text. |

These are typical choices; suitably trained models from other families can also perform several of these tasks.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [ ]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]
sentiment_results = classifier(sentences)

for sentence, result in zip(sentences, sentiment_results):
    print("Sentence:", sentence)
    print("Predicted label:", result["label"])
    print(f"Score: {result['score']:.4f}")
    print()

The score is the model's confidence for its predicted label, not a guarantee that the prediction is correct. This SST-2 model predicts positive or negative sentiment; it does not have a neutral class.

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


### Summarize a Short Paragraph
Prefix the paragraph with `Summarize:` to specify the instruction. Use deterministic decoding and print the model-generated summary alongside the original text.

In [ ]:
paragraph = (
    "Natural language processing helps computers work with human language. "
    "It is used for sentiment analysis, translation, and summarization. "
    "Transformer models use attention to represent relationships between tokens. "
    "Pretrained models can be adapted to specific language tasks, but their outputs "
    "still need evaluation because they can contain errors."
)
summarization_prompt = "Summarize: " + paragraph
summary_inputs = tokenizer(
    summarization_prompt, return_tensors="pt", truncation=True, max_length=512
)
with torch.no_grad():
    summary_ids = model.generate(
        **summary_inputs, max_new_tokens=60, num_beams=4, do_sample=False
    )
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original paragraph:", paragraph)
print("\nGenerated summary:", summary)


## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


A decoder-only model is suitable for a chatbot because it generates a response one token at a time using the conversation supplied as context. Causal attention lets each generated token depend on earlier tokens, including the user's question and the response so far. Training on large text collections teaches language patterns, and instruction tuning helps the model follow requests and produce useful conversational responses. Its fluent output can still contain factual errors, so fluency alone does not guarantee correctness.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [ ]:
sampling_prompt = "Natural language processing is"

# Do not reset the random seed inside this loop.
for run in range(3):
    sampled_output = generator(
        sampling_prompt,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"Run {run + 1}:", sampled_output[0]["generated_text"])

With sampling enabled, the model chooses tokens randomly from a probability distribution instead of always choosing the most likely token. Each choice changes the context for the next step, so outputs can differ even with the same prompt. The setup seed makes a sequence of runs more reproducible, but the random state advances between calls; resetting the seed before every call would usually reproduce the same output.